# CS431 — Thử nghiệm huấn luyện Hướng A (Learned Reference Selector) & Hướng B (Exposure-Consistency LoRA)

- **Tách dữ liệu nghiêm ngặt (không rò rỉ)**:
  - **Tập huấn luyện (4 cảnh)**: `Ujikintoki`, `BlueHawaii`, `Chocolate`, `Cupcake`.
  - **Tập kiểm thử giữ lại (5 cảnh chưa từng thấy khi train)**: `GearWorks`, `Laboratory`, `MilkCookie`, `Popcorn`, `Sculpture`.
- **Hướng A (`LearnedReferenceScorer`)**: Mạng xếp hạng hoán vị-đẳng biến (~6K tham số) kết hợp đặc trưng RGB (phơi sáng, cầu nối phơi sáng, SIFT overlap, độ sắc nét) + đặc trưng DINOv2 CLS token trích trực tiếp từ `DA3-Base` đóng băng, huấn luyện bằng listwise KL loss theo nhãn mềm `softmax(F1 / tau)` trên 48 tập 4 góc nhìn sinh từ 4 cảnh train.
- **Hướng B (`Exposure-Consistency LoRA`)**: Gắn LoRA (`rank=4`, `alpha=8`, ~147K tham số = 0.12% DA3-Base) vào `qkv`/`proj` của 6 block cuối DINOv2; huấn luyện tự giám sát thầy-trò (không cần nhãn 3D) giữa ảnh `varyexp` (trò) và `uniform_mid` (thầy đóng băng), kèm 20% bước chuẩn hóa đồng nhất trên ảnh sạch.
- **So sánh công bằng 6 biến thể trên 5 cảnh held-out (`resolution=504`, `4 views`, `colmap_sparse + c2w`)**:
  1. `base_default` (DA3 gốc + `saddle_balanced`)
  2. `base_combined` (DA3 gốc + Heuristic v2)
  3. `base_learned_A` (DA3 gốc + Hướng A)
  4. `lora_default_B` (Hướng B LoRA + `saddle_balanced`)
  5. `lora_combined_B` (Hướng B LoRA + Heuristic v2)
  6. `lora_learned_AB` (Hướng A + Hướng B)


In [ ]:
import csv
import json
import os
import random
import re
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path
from time import perf_counter

import numpy as np
from PIL import Image

WORK_DIR = Path('/kaggle/working')
OUT_DIR = WORK_DIR / 'outputs'
TMP_DIR = Path('/tmp/cs431_ab')
REPO_DIR = TMP_DIR / 'Depth-Anything-3'
DATA_DIR = TMP_DIR / 'realx3d'
PC_DIR = TMP_DIR / 'pointclouds'

DA3_REPO_URL = 'https://github.com/ByteDance-Seed/Depth-Anything-3.git'
DA3_COMMIT = '3d835ec1a5802d64a8b8b15f817a1ab54809bfe4'
HF_REALX3D_BASE = 'https://huggingface.co/datasets/ToferFish/RealX3D/resolve/main'
VARYEXP_TAR_URL = f'{HF_REALX3D_BASE}/data_4/data4_varyexp.tar.gz'

TRAIN_SCENES = ['Ujikintoki', 'BlueHawaii', 'Chocolate', 'Cupcake']
HELDOUT_SCENES = ['GearWorks', 'Laboratory', 'MilkCookie', 'Popcorn', 'Sculpture']
ALL_SCENES = TRAIN_SCENES + HELDOUT_SCENES
NUM_VIEWS = 4
EVAL_RES = 504
TRAIN_LORA_RES = 336
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

for d in (OUT_DIR, TMP_DIR, DATA_DIR, PC_DIR):
    d.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=None, env=None, check=True):
    print(f"$ {' '.join(map(str, cmd))}", flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

def locate_cs431_code() -> Path:
    for found in Path('/kaggle/input').rglob('run_experiment.py'):
        return found.parent
    raise FileNotFoundError('Could not locate cs431/run_experiment.py under /kaggle/input')

run_cmd(['nvidia-smi'], check=False)
if not REPO_DIR.exists():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    run_cmd(['git', 'init'], cwd=REPO_DIR)
    run_cmd(['git', 'remote', 'add', 'origin', DA3_REPO_URL], cwd=REPO_DIR)
    run_cmd(['git', 'fetch', '--depth', '1', 'origin', DA3_COMMIT], cwd=REPO_DIR)
    run_cmd(['git', 'checkout', 'FETCH_HEAD'], cwd=REPO_DIR)

cs431_src = locate_cs431_code()
cs431_dst = REPO_DIR / 'cs431'
if cs431_dst.exists():
    shutil.rmtree(cs431_dst)
shutil.copytree(cs431_src, cs431_dst)

pkgs = [
    'addict', 'omegaconf', 'einops>=0.8', 'huggingface_hub', 'safetensors',
    'evo', 'plyfile', 'trimesh', 'e3nn', 'moviepy==1.0.3', 'imageio', 'pycolmap', 'pillow_heif'
]
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

os.environ['HF_HOME'] = str(TMP_DIR / 'hf_cache')
for p in (str(REPO_DIR / 'src'), str(REPO_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

env = os.environ.copy()
env['PYTHONPATH'] = f"{REPO_DIR / 'src'}:{REPO_DIR}"
run_cmd([sys.executable, '-m', 'unittest', 'cs431.test_experiment', '-v'], cwd=REPO_DIR, env=env)


## Bước 2: Stream dữ liệu 9 cảnh RealX3D (4 train + 5 held-out) một lần duy nhất


In [ ]:
def stream_extract_varyexp(target_scenes: list[str]):
    target_set = set(target_scenes)
    extracted_files = 0
    extracted_bytes = 0
    t0 = perf_counter()
    print(f'Streaming {VARYEXP_TAR_URL} for {len(target_scenes)} scenes...', flush=True)
    proc = subprocess.Popen(
        ['curl', '-fsSL', '--retry', '5', '--retry-delay', '5', VARYEXP_TAR_URL],
        stdout=subprocess.PIPE,
    )
    try:
        with tarfile.open(fileobj=proc.stdout, mode='r|gz') as tar:
            for member in tar:
                if not member.isfile():
                    continue
                parts = Path(member.name).parts
                if len(parts) < 3:
                    continue
                scene = parts[1] if parts[0] == 'varyexp' else parts[0]
                rel_parts = parts[2:] if parts[0] == 'varyexp' else parts[1:]
                if scene not in target_set or not rel_parts:
                    continue
                top = rel_parts[0]
                is_meta = top in ('transforms_train.json', 'colmap2world.txt')
                is_sparse = top == 'sparse' and Path(member.name).name in ('images.bin', 'images.txt')
                is_train_img = top.startswith('train') and member.name.lower().endswith(('.jpg', '.jpeg', '.png'))
                if not (is_meta or is_sparse or is_train_img):
                    continue
                dst = DATA_DIR / scene / Path(*rel_parts)
                dst.parent.mkdir(parents=True, exist_ok=True)
                src_f = tar.extractfile(member)
                if src_f is not None:
                    with dst.open('wb') as out_f:
                        shutil.copyfileobj(src_f, out_f)
                    extracted_files += 1
                    extracted_bytes += member.size
                    if extracted_files % 250 == 0:
                        print(f'  extracted {extracted_files} files ({extracted_bytes / 1e9:.2f} GB) in {perf_counter() - t0:.1f}s', flush=True)
    finally:
        if proc.stdout:
            proc.stdout.close()
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'curl exited with status {ret}')
    print(f'Done streaming: {extracted_files} files ({extracted_bytes / 1e9:.2f} GB) in {perf_counter() - t0:.1f}s', flush=True)

def download_pointcloud(scene: str) -> Path:
    url = f'{HF_REALX3D_BASE}/pointclouds/{scene}.tar.gz'
    scene_pc_dir = PC_DIR / scene
    scene_pc_dir.mkdir(parents=True, exist_ok=True)
    ply_path = None
    proc = subprocess.Popen(
        ['curl', '-fsSL', '--retry', '5', '--retry-delay', '5', url],
        stdout=subprocess.PIPE,
    )
    try:
        with tarfile.open(fileobj=proc.stdout, mode='r|gz') as tar:
            for member in tar:
                if member.isfile() and Path(member.name).name == 'cull_pointcloud.ply':
                    dst = scene_pc_dir / 'cull_pointcloud.ply'
                    src_f = tar.extractfile(member)
                    if src_f is not None:
                        with dst.open('wb') as out_f:
                            shutil.copyfileobj(src_f, out_f)
                    ply_path = dst
    finally:
        if proc.stdout:
            proc.stdout.close()
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'curl failed for {url} (exit {ret})')
    return ply_path

stream_extract_varyexp(ALL_SCENES)


## Bước 3 (Hướng A): Tạo dữ liệu huấn luyện & train `LearnedReferenceScorer` trên 4 cảnh train


In [ ]:
import torch
from plyfile import PlyData
from depth_anything_3.api import DepthAnything3
from cs431.geometry import cloud_metrics, predicted_poses, relative_pose_metrics, to_metric_poses, voxel_sample
from cs431.run_experiment import DA3_BASE, DA3_BASE_REVISION, load_colmap_views
from cs431.learned_selector import extract_view_features, train_reference_scorer
from cs431.lora_adapt import attach_lora_to_da3, exposure_consistency_loss, save_lora_weights, set_lora_enabled

torch.manual_seed(SEED)
device = torch.device('cuda:0')
assert torch.cuda.is_available(), 'CUDA is required'
print('Loading DA3-Base on', torch.cuda.get_device_name(0), flush=True)
model = DepthAnything3.from_pretrained(DA3_BASE, revision=DA3_BASE_REVISION).to(device).eval()

def load_gt_cloud(ply_path: Path) -> np.ndarray:
    vertices = PlyData.read(str(ply_path))['vertex'].data
    xyz = np.column_stack([vertices[a] for a in 'xyz']).astype(float)
    return voxel_sample(xyz)

def _exposure_sort_key(folder_name: str):
    m = re.search(r'(\d+)', folder_name)
    return (0, int(m.group(1))) if m else (1, folder_name)

def get_scene_view_sets(scene_dir: Path, offsets=(0, 2, 4, 6), num_views: int = NUM_VIEWS):
    frames = json.loads((scene_dir / 'transforms_train.json').read_text(encoding='utf-8'))['frames']
    train_dirs = sorted(
        [p for p in scene_dir.iterdir() if p.is_dir() and p.name.startswith('train')],
        key=lambda p: _exposure_sort_key(p.name),
    )
    iso_dirs = [p for p in train_dirs if re.search(r'\d+', p.name)]
    sets = []
    for k, off in enumerate(offsets):
        start_idx = min(off, max(0, len(frames) - num_views))
        end_idx = min(len(frames) - 1, max(start_idx + num_views - 1, len(frames) - 1 - k))
        idxs = np.linspace(start_idx, end_idx, num_views, dtype=int)
        sampled = [frames[i] for i in idxs]
        names = [Path(f['file_path']).name for f in sampled]
        default_paths = [scene_dir / f['file_path'] for f in sampled]
        if all(p.is_file() for p in default_paths):
            sets.append((f'default_off{off}', default_paths))
        complete_iso = [d for d in iso_dirs if all((d / n).is_file() for n in names)]
        if len(complete_iso) >= 2:
            picks = np.linspace(0, len(complete_iso) - 1, num_views, dtype=int)
            # Rotate exposure assignment per offset for diversity
            picks = np.roll(picks, off % num_views)
            sets.append((f'vary_off{off}', [complete_iso[pi] / n for pi, n in zip(picks, names)]))
            mid_dir = complete_iso[len(complete_iso) // 2]
            sets.append((f'uniform_off{off}', [mid_dir / n for n in names]))
    return sets

t_a0 = perf_counter()
train_samples = []
for scene in TRAIN_SCENES:
    scene_dir = DATA_DIR / scene
    ply_path = download_pointcloud(scene)
    gt_points = load_gt_cloud(ply_path)
    c2w_mat = np.loadtxt(scene_dir / 'colmap2world.txt')
    view_sets = get_scene_view_sets(scene_dir, offsets=(0, 2, 4, 6))
    print(f'[Dir A] Collecting {len(view_sets)} training sets from {scene}...', flush=True)
    for tag, img_paths in view_sets:
        paths, gt_poses = load_colmap_views(scene_dir / 'sparse' / '0', img_paths)
        gt_metric = to_metric_poses(gt_poses, c2w_mat)
        imgs = [Image.open(p).convert('RGB') for p in paths]
        feats, _ = extract_view_features(imgs, model=model, process_res=336)
        f1s = []
        for ref_i in range(NUM_VIEWS):
            order = [ref_i] + [j for j in range(NUM_VIEWS) if j != ref_i]
            ordered = [imgs[j] for j in order]
            pred = model.inference(ordered, ref_view_strategy='first', process_res=EVAL_RES)
            m = cloud_metrics(pred, gt_metric[order], gt_points)
            f1s.append(float(m['f1']) if m.get('f1') is not None else 0.0)
        train_samples.append((feats, np.asarray(f1s, dtype=np.float32)))
    shutil.rmtree(PC_DIR / scene, ignore_errors=True)

scorer_A, loss_hist_A = train_reference_scorer(train_samples, epochs=300, lr=3e-3, weight_decay=2e-2, tau=0.04, seed=SEED)
torch.save(scorer_A.state_dict(), OUT_DIR / 'learned_selector_A.pt')
print(f'[Dir A] Trained on {len(train_samples)} sets in {perf_counter() - t_a0:.1f}s | loss: {loss_hist_A[0]:.4f} -> {loss_hist_A[-1]:.4f}', flush=True)


## Bước 4 (Hướng B): Huấn luyện Exposure-Consistency LoRA (`rank=4`) trên 4 cảnh train
Chưng cất tự giám sát thầy-trò (không cần nhãn 3D): thầy là `DA3-Base` gốc (tắt LoRA) chạy trên ảnh `uniform_mid`, trò là `DA3-Base + LoRA` chạy trên ảnh `varyexp` cùng 4 góc nhìn (20% bước đồng nhất trên ảnh sạch).


In [ ]:
t_b0 = perf_counter()
lora_params = attach_lora_to_da3(model, rank=4, alpha=8.0, start_block=6)
n_lora = sum(p.numel() for p in lora_params)
n_total = sum(p.numel() for p in model.parameters())
print(f'[Dir B] Attached LoRA: {n_lora:,} trainable / {n_total:,} total ({100.0 * n_lora / n_total:.3f}%)', flush=True)

# Build paired (varyexp, clean) image path pools from the 4 train scenes
train_scene_pools = []
for scene in TRAIN_SCENES:
    sdir = DATA_DIR / scene
    frames = json.loads((sdir / 'transforms_train.json').read_text(encoding='utf-8'))['frames']
    names = [Path(f['file_path']).name for f in frames]
    iso_dirs = sorted(
        [p for p in sdir.iterdir() if p.is_dir() and re.search(r'\d+', p.name)],
        key=lambda p: _exposure_sort_key(p.name),
    )
    clean_dir = iso_dirs[len(iso_dirs) // 2]
    all_vary_dirs = iso_dirs + ([sdir / 'train'] if (sdir / 'train').is_dir() else [])
    train_scene_pools.append((scene, names, clean_dir, all_vary_dirs))

LORA_STEPS = 160
GRAD_ACCUM = 4
opt_b = torch.optim.AdamW(lora_params, lr=1e-4, weight_decay=1e-2)
sched_b = torch.optim.lr_scheduler.CosineAnnealingLR(opt_b, T_max=LORA_STEPS // GRAD_ACCUM)
scaler = torch.amp.GradScaler('cuda')

model.eval()  # Keep frozen BN/Dropout in eval mode while training LoRA weights
opt_b.zero_grad()
running_losses = []
for step in range(1, LORA_STEPS + 1):
    scene, names, clean_dir, vary_dirs = random.choice(train_scene_pools)
    # Sample 4 sorted frame indices spanning the trajectory
    idxs = sorted(random.sample(range(len(names)), NUM_VIEWS))
    chosen_names = [names[i] for i in idxs]
    clean_imgs = [Image.open(clean_dir / n).convert('RGB') for n in chosen_names]

    if random.random() < 0.20:
        # Identity regularization on clean views
        vary_imgs = clean_imgs
    else:
        vary_imgs = [Image.open(random.choice(vary_dirs) / n).convert('RGB') for n in chosen_names]

    # Teacher pass (frozen base DA3, LoRA disabled)
    set_lora_enabled(model, False)
    with torch.no_grad():
        c_cpu, _, _ = model._preprocess_inputs(clean_imgs, None, None, process_res=TRAIN_LORA_RES)
        c_t, _, _ = model._prepare_model_inputs(c_cpu, None, None)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            pred_t = model.model(c_t, ref_view_strategy='first')

    # Student pass (LoRA enabled)
    set_lora_enabled(model, True)
    v_cpu, _, _ = model._preprocess_inputs(vary_imgs, None, None, process_res=TRAIN_LORA_RES)
    v_t, _, _ = model._prepare_model_inputs(v_cpu, None, None)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        pred_s = model.model(v_t, ref_view_strategy='first')
        loss, stats = exposure_consistency_loss(pred_s, pred_t)
        loss_scaled = loss / GRAD_ACCUM

    scaler.scale(loss_scaled).backward()
    running_losses.append(stats['total'])
    if step % GRAD_ACCUM == 0:
        scaler.unscale_(opt_b)
        torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
        scaler.step(opt_b)
        scaler.update()
        opt_b.zero_grad()
        sched_b.step()
    if step % 40 == 0:
        peak_gb = torch.cuda.max_memory_allocated(device) / 1024**3
        print(f'  step {step:3d}/{LORA_STEPS} | mean_loss={np.mean(running_losses[-40:]):.4f} | silog={stats["silog"]:.4f} | grad={stats["grad"]:.4f} | pose={stats["pose"]:.4f} | peak_vram={peak_gb:.2f}GB', flush=True)

save_lora_weights(model, OUT_DIR / 'da3_lora_r4.pt')
print(f'[Dir B] LoRA training complete in {perf_counter() - t_b0:.1f}s', flush=True)
torch.cuda.empty_cache()


## Bước 5: Đánh giá công bằng 6 biến thể trên 5 cảnh giữ lại (Held-out, `resolution = 504`)


In [ ]:
def build_eval_conditions(scene_dir: Path, num_views: int = NUM_VIEWS):
    frames = json.loads((scene_dir / 'transforms_train.json').read_text(encoding='utf-8'))['frames']
    idxs = np.linspace(0, len(frames) - 1, num_views, dtype=int)
    sampled = [frames[i] for i in idxs]
    names = [Path(f['file_path']).name for f in sampled]
    iso_dirs = sorted(
        [p for p in scene_dir.iterdir() if p.is_dir() and re.search(r'\d+', p.name)],
        key=lambda p: _exposure_sort_key(p.name),
    )
    conds = {'default_json': [scene_dir / f['file_path'] for f in sampled]}
    complete_iso = [d for d in iso_dirs if all((d / n).is_file() for n in names)]
    picks = np.linspace(0, len(complete_iso) - 1, num_views, dtype=int)
    conds['varyexp_mixed'] = [complete_iso[pi] / n for pi, n in zip(picks, names)]
    mid_dir = complete_iso[len(complete_iso) // 2]
    conds['uniform_mid'] = [mid_dir / n for n in names]
    return conds

@torch.inference_mode()
def eval_one(images, gt_poses, gt_metric, gt_points, lora_on: bool, strategy: str, ref_idx: int | None = None):
    set_lora_enabled(model, lora_on)
    if ref_idx is not None:
        order = [ref_idx] + [i for i in range(len(images)) if i != ref_idx]
        ordered = [images[i] for i in order]
        pred = model.inference(ordered, ref_view_strategy='first', process_res=EVAL_RES)
    else:
        order = list(range(len(images)))
        pred = model.inference(images, ref_view_strategy=strategy, process_res=EVAL_RES)
    rel = relative_pose_metrics(predicted_poses(pred.extrinsics), gt_poses[order])
    cm = cloud_metrics(pred, gt_metric[order], gt_points)
    return {'f1': round(cm['f1'], 4), 'auc10': round(rel['auc10'], 4), 'med_m': round(cm['median_pred_to_gt_m'], 4)}

rows = []
for scene in HELDOUT_SCENES:
    scene_dir = DATA_DIR / scene
    ply_path = download_pointcloud(scene)
    gt_points = load_gt_cloud(ply_path)
    c2w_mat = np.loadtxt(scene_dir / 'colmap2world.txt')
    conds = build_eval_conditions(scene_dir)

    for cond_name, img_paths in conds.items():
        paths, gt_poses = load_colmap_views(scene_dir / 'sparse' / '0', img_paths)
        gt_metric = to_metric_poses(gt_poses, c2w_mat)
        imgs = [Image.open(p).convert('RGB') for p in paths]

        # Extract features with base DA3 for fair reference selection
        set_lora_enabled(model, False)
        feats, heur_scores = extract_view_features(imgs, model=model, process_res=336)
        ref_heur = int(np.argmax(heur_scores['combined']))
        with torch.no_grad():
            ref_learned = int(torch.argmax(scorer_A(torch.from_numpy(feats).float())).item())

        r_base_def = eval_one(imgs, gt_poses, gt_metric, gt_points, False, 'saddle_balanced', None)
        r_base_comb = eval_one(imgs, gt_poses, gt_metric, gt_points, False, 'first', ref_heur)
        r_base_A = r_base_comb if ref_learned == ref_heur else eval_one(imgs, gt_poses, gt_metric, gt_points, False, 'first', ref_learned)

        r_lora_def = eval_one(imgs, gt_poses, gt_metric, gt_points, True, 'saddle_balanced', None)
        r_lora_comb = eval_one(imgs, gt_poses, gt_metric, gt_points, True, 'first', ref_heur)
        r_lora_AB = r_lora_comb if ref_learned == ref_heur else eval_one(imgs, gt_poses, gt_metric, gt_points, True, 'first', ref_learned)

        row = {
            'scene': scene,
            'condition': cond_name,
            'ref_heur': ref_heur,
            'ref_learned': ref_learned,
            'base_def_f1': r_base_def['f1'], 'base_def_auc10': r_base_def['auc10'],
            'base_comb_f1': r_base_comb['f1'], 'base_comb_auc10': r_base_comb['auc10'],
            'base_A_f1': r_base_A['f1'], 'base_A_auc10': r_base_A['auc10'],
            'lora_def_f1': r_lora_def['f1'], 'lora_def_auc10': r_lora_def['auc10'],
            'lora_comb_f1': r_lora_comb['f1'], 'lora_comb_auc10': r_lora_comb['auc10'],
            'lora_AB_f1': r_lora_AB['f1'], 'lora_AB_auc10': r_lora_AB['auc10'],
        }
        rows.append(row)
        print(
            f"{scene:11s} {cond_name:14s} | F1: BaseDef={row['base_def_f1']} BaseComb={row['base_comb_f1']} "
            f"Base+A={row['base_A_f1']} LoRA+Def={row['lora_def_f1']} LoRA+Comb={row['lora_comb_f1']} LoRA+AB={row['lora_AB_f1']}",
            flush=True,
        )
    shutil.rmtree(PC_DIR / scene, ignore_errors=True)

with (OUT_DIR / 'summary_ab.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

md = [
    '# CS431 Training Experiment: Direction A (Learned Selector) & Direction B (LoRA)',
    '',
    '## Mean F1 @ 5cm on 5 Held-out Scenes',
    '| Condition | Base+Def | Base+Comb (v2) | Base+A (Learned) | LoRA+Def (B) | LoRA+Comb (B+v2) | LoRA+AB |',
    '| --- | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for cond in ('default_json', 'varyexp_mixed', 'uniform_mid'):
    sub = [r for r in rows if r['condition'] == cond]
    m = lambda k: round(float(np.mean([r[k] for r in sub])), 4)
    md.append(f"| {cond} | {m('base_def_f1')} | {m('base_comb_f1')} | {m('base_A_f1')} | {m('lora_def_f1')} | {m('lora_comb_f1')} | {m('lora_AB_f1')} |")

md += ['', '## Mean AUC@10 on 5 Held-out Scenes', '| Condition | Base+Def | Base+Comb (v2) | Base+A (Learned) | LoRA+Def (B) | LoRA+Comb (B+v2) | LoRA+AB |', '| --- | ---: | ---: | ---: | ---: | ---: | ---: |']
for cond in ('default_json', 'varyexp_mixed', 'uniform_mid'):
    sub = [r for r in rows if r['condition'] == cond]
    m = lambda k: round(float(np.mean([r[k] for r in sub])), 4)
    md.append(f"| {cond} | {m('base_def_auc10')} | {m('base_comb_auc10')} | {m('base_A_auc10')} | {m('lora_def_auc10')} | {m('lora_comb_auc10')} | {m('lora_AB_auc10')} |")

md_text = '\n'.join(md) + '\n'
(OUT_DIR / 'summary_ab.md').write_text(md_text, encoding='utf-8')
print('\n' + md_text)
